# 11.5 Join Strategies Overview

## Learning Objectives

By the end of this lesson you will be able to:

- Understand why joins are expensive in Spark
- Learn different Spark join strategies
- Understand Broadcast Join
- Understand Sort Merge Join
- Understand Shuffle Hash Join
- Learn when Spark chooses each strategy
- Read join operators in execution plans

> **Core Rule:** Join optimization is one of the highest-impact Spark performance skills.

## Setup: Initialize Spark & Mock Data

Let's start our local Spark Session and generate two datasets: a "small" Customers table and a "large" Orders table. We will use these to demonstrate how Spark changes its join strategy.

In [ ]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

# Initialize a local Spark session
spark = SparkSession.builder \
    .appName("Join_Strategies") \
    .master("local[*]") \
    .getOrCreate()

# 1. Create a "Small" Customers Table
customers_data = [(1, "Alice", "New York"), (2, "Bob", "London"), (3, "Charlie", "Paris")]
customers_df = spark.createDataFrame(customers_data, ["customer_id", "name", "city"])

# 2. Create a "Large" Orders Table
orders_data = [(101, 1, 250.0), (102, 2, 150.0), (103, 1, 300.0), (104, 3, 500.0), (105, 2, 120.0)]
orders_df = spark.createDataFrame(orders_data, ["order_id", "customer_id", "amount"])

print("Spark Session Initialized and Mock Data Created!")

# Why Are Joins Important?

In real-world Data Engineering projects, joins are everywhere.

Examples:
- Orders + Customers
- Transactions + Accounts
- Products + Categories
- Clickstream + User Profiles

Most business analytics pipelines depend heavily on joins. Unfortunately, joins are also among the most expensive Spark operations.

# Why Can Joins Become Slow?

To perform a join, Spark often needs to:

- Move data between executors
- Shuffle records
- Sort data
- Compare matching keys

These operations consume CPU, Memory, Disk, and Network bandwidth. The larger the datasets, the greater the cost.

# Spark Doesn't Use One Join Method

Many beginners assume: *"There is only one way to perform a join."*

In reality, Spark can choose different strategies. Common join strategies include:

1. **Broadcast Join**
2. **Sort Merge Join**
3. **Shuffle Hash Join**

Each strategy has different performance characteristics.

<h3>Join Strategy Selection</h3>

<img src="../assets/Module_11/11_05_01.png" alt="Join Strategy Selection" width="75%">

<p><i><b>Image Prompt:</b> Apache Spark join strategy decision diagram showing Broadcast Join, Sort Merge Join and Shuffle Hash Join. Spark optimizer selecting the best strategy based on dataset size. Educational infographic.</i></p>

# 1. Broadcast Join Introduction

Broadcast Join is usually the fastest join strategy.

Spark sends the smaller dataset to every executor. Each executor performs the join locally. This avoids large-scale data movement (Shuffle).

**How it works:**
1. Identify the small table.
2. Copy the small table to all executors.
3. Join locally with the large table.

Let's see this in action.

In [ ]:
# Because our customers_df is tiny, Spark will naturally choose a Broadcast Join.
broadcast_join = orders_df.join(customers_df, "customer_id")

print("Execution Plan for Broadcast Join:")
print("-"*50)
# Look for 'BroadcastHashJoin' in the output
broadcast_join.explain()
print("-"*50)
print("Notice that there is NO 'Exchange' operator. Data was not shuffled!")

<h3>Broadcast Join Architecture</h3>

<img src="../assets/Module_11/11_05_02.png" alt="Broadcast Join Architecture" width="75%">

<p><i><b>Image Prompt:</b> Apache Spark Broadcast Join visualization. Small lookup table copied to all executors while large dataset remains distributed. Local joins occurring without shuffle. Educational architecture diagram.</i></p>

# Limitations of Broadcast Join

Broadcast Join is not always possible. Problems occur when:

- The small table becomes large (default limit is 10MB)
- Executor memory is insufficient
- Broadcast size exceeds thresholds

Large datasets should not be broadcast, or you will get an OutOfMemory error.

# 2. Sort Merge Join Introduction

When datasets become large, Spark commonly chooses **Sort Merge Join**. This is one of the most frequently used join strategies in production systems.

**How it works:**
1. Shuffle data by join key.
2. Sort records inside partitions.
3. Merge matching records.

Because sorting and shuffling are required, this strategy is more expensive. Let's force Spark to use this strategy by disabling Broadcast joins.

In [ ]:
# Force Spark to NOT use Broadcast Joins by setting the threshold to -1
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")

smj_join = orders_df.join(customers_df, "customer_id")

print("Execution Plan for Sort Merge Join:")
print("-"*50)
# Look for 'SortMergeJoin', 'Sort', and 'Exchange' in the output
smj_join.explain()
print("-"*50)
print("Notice the 'Exchange' (Shuffle) and 'Sort' operators before the Join happens.")

<h3>Sort Merge Join Workflow</h3>

<img src="../assets/Module_11/11_05_04.png" alt="Sort Merge Join Workflow" width="75%">

<p><i><b>Image Prompt:</b> Apache Spark Sort Merge Join process. Data shuffled by join key, sorted within partitions and then merged. Clear educational architecture diagram with arrows.</i></p>

# 3. Shuffle Hash Join Introduction

Another join strategy is **Shuffle Hash Join**.

This strategy combines a Shuffle with hash-based matching instead of sorting records.

**How it works:**
1. Shuffle both datasets.
2. Build hash tables on the smaller of the two shuffled partitions.
3. Match records using hash lookups.

Hash lookups can be faster than sorting, but they consume high executor memory. Let's force Spark to use this.

In [ ]:
# Tell Spark we DO NOT prefer Sort Merge Joins
spark.conf.set("spark.sql.join.preferSortMergeJoin", "false")

shj_join = orders_df.join(customers_df, "customer_id")

print("Execution Plan for Shuffle Hash Join:")
print("-"*50)
# Look for 'ShuffledHashJoin' and 'Exchange' in the output
shj_join.explain()
print("-"*50)
print("Notice the 'ShuffledHashJoin' operator. Data is shuffled (Exchange), but NOT sorted!")

# Reset configurations back to defaults for best practices
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "10485760")
spark.conf.set("spark.sql.join.preferSortMergeJoin", "true")

<h3>Shuffle Hash Join Workflow</h3>

<img src="../assets/Module_11/11_05_06.png" alt="Shuffle Hash Join Workflow" width="75%">

<p><i><b>Image Prompt:</b> Apache Spark Shuffle Hash Join architecture. Datasets shuffled into partitions followed by hash table creation and hash-based matching. Educational diagram.</i></p>

# Comparing Join Strategies

| Strategy | Shuffle | Sorting | Broadcast | Typical Speed |
|-----------|----------|----------|-----------|---------------|
| **Broadcast Join** | Minimal | No | Yes | Fastest |
| **Sort Merge Join** | Yes | Yes | No | Moderate |
| **Shuffle Hash Join** | Yes | No | No | Moderate |

<h3>Join Strategy Comparison</h3>
<img src="../assets/Module_11/11_05_08.png" alt="Join Strategy Comparison" width="75%">
<p><i><b>Image Prompt:</b> Side-by-side comparison of Broadcast Join, Sort Merge Join and Shuffle Hash Join. Comparing shuffle, sorting, memory usage and performance. Professional educational infographic.</i></p>

# Real-World Example

Suppose we have a Customers Table (50,000 rows) and an Orders Table (200 Million rows).

**Broadcast Join:**
Execution Time = 3 Minutes

**Sort Merge Join:**
Execution Time = 12 Minutes

Same result. Different strategy. Huge performance difference.

# Key Takeaways

- Spark supports multiple join strategies.
- **Broadcast Join** is usually fastest for small lookup tables.
- **Sort Merge Join** is common for large datasets and involves sorting.
- **Shuffle Hash Join** uses hash-based matching instead of sorting.
- Join strategy selection directly affects performance.
- Execution plans reveal which join strategy Spark selected.

---

# Next Lesson: 11.6 Broadcast Joins

In the next lesson we will do a deep dive into Broadcast joins, exploring:
- Exactly when to use them
- Internal Benefits
- Memory Limitations

Broadcast Join is one of the most practical Spark optimization techniques used in production.